In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Modo de prueba local de Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Usa el modo de prueba local (disponible con `qiskit-ibm-runtime` v0.22.0 o posterior) para probar programas antes de ajustarlos y enviarlos a hardware cuántico real. Después de usar el modo de prueba local para verificar tu programa, lo único que necesitas cambiar es el nombre del backend para ejecutarlo en una QPU.

Para usar el modo de prueba local, especifica uno de los fake backends de ``qiskit_ibm_runtime.fake_provider`` o especifica un backend de Qiskit Aer al instanciar una primitiva de Qiskit Runtime o una sesión.

- **Fake backends**: Los [fake backends](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) en ``qiskit_ibm_runtime.fake_provider`` imitan el comportamiento de las QPUs de IBM&reg; mediante el uso de instantáneas de QPU. Las instantáneas de QPU contienen información importante sobre la QPU, como el mapa de acoplamiento, las gates de base y las propiedades de los qubits, que son útiles para probar el Transpiler y realizar simulaciones con ruido de la QPU. El modelo de ruido de la instantánea se aplica automáticamente durante la simulación.
- **Simulador Aer**: Los simuladores de [Qiskit Aer](/guides/simulate-with-qiskit-aer) ofrecen simulación de mayor rendimiento, capaz de manejar Circuits más grandes y [modelos de ruido personalizados](/guides/build-noise-models). Cuando usas `AerSimulator` en el modo de prueba local, tienes disponible una lista de opciones de métodos de simulación. Consulta el [ejemplo del modo de simulación Clifford](#clifford-sim), que demuestra cómo simular eficientemente Circuits de Clifford con un gran número de qubits.

    <details>
    <summary>**Lista de métodos de simulación disponibles en Qiskit Aer**</summary>

    Consulta la documentación de [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) para más información.

    * ``"automatic"``: Método de simulación predeterminado. Selecciona el método de simulación automáticamente según el Circuit y el modelo de ruido.

    * ``"statevector"``: Una simulación de vector de estado denso que puede muestrear resultados de medición de Circuits *ideales* con todas las mediciones al final del Circuit. En simulaciones con ruido, cada disparo muestrea un Circuit ruidoso seleccionado aleatoriamente del modelo de ruido.

    * ``"density_matrix"``: Una simulación de matriz de densidad que puede muestrear resultados de medición de Circuits *ruidosos* con todas las mediciones al final del Circuit.

    * ``"stabilizer"``: Un simulador eficiente de estados estabilizadores de Clifford que puede simular Circuits de Clifford con ruido si todos los errores en el modelo de ruido también son errores de Clifford.

    * ``"extended_stabilizer"``: Un simulador aproximado para Circuits de Clifford + T basado en descomponer el estado en un estado estabilizador de rango. El número de términos crece con el número de gates no-Clifford (T).

    * ``"matrix_product_state"``: Un simulador de vector de estado en red tensorial que usa una representación de Estado Producto Matricial (MPS) para el estado. Esto puede hacerse con o sin truncar las dimensiones de enlace del MPS, según las opciones del simulador. El comportamiento predeterminado es sin truncamiento.

    * ``"unitary"``: Una simulación de matriz unitaria densa de un Circuit ideal. Simula la matriz unitaria del Circuit en sí, en lugar de la evolución de un estado cuántico inicial. Este método solo puede simular gates; no admite medición, reinicio ni ruido.

    * ``"superop"``: Una simulación de matriz de superoperador densa de un Circuit ideal o ruidoso. Simula la matriz de superoperador del Circuit en sí, en lugar de la evolución de un estado cuántico inicial. Este método puede simular gates y reinicios ideales y ruidosos, pero no admite mediciones.

    * ``"tensor_network"``: Una simulación basada en redes tensoriales que admite tanto vector de estado como matriz de densidad. Actualmente solo está disponible para GPU y se acelera usando las APIs `cuTensorNet` de cuQuantum.
    </details>

> **Note:** - Puedes especificar todas las opciones de Qiskit Runtime en el modo de prueba local. Sin embargo, todas las opciones excepto shots se ignoran al ejecutarse en un simulador local.
> - Se recomienda instalar Qiskit Aer antes de usar fake backends o simuladores Aer ejecutando `pip install qiskit-aer`. Los fake backends usan simuladores Aer internamente cuando están disponibles, para aprovechar su rendimiento.


## Ejemplo con fake backends

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Ejemplos con AerSimulator {#aerSimulator-examples}

Ejemplo con sesiones, sin ruido:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Para simular con ruido, especifica una QPU (hardware cuántico) y envíala a Aer. Aer construye un modelo de ruido basado en los datos de calibración de esa QPU e instancia un backend Aer con ese modelo. Si lo prefieres, puedes [construir un modelo de ruido](/guides/build-noise-models).

> **Caution:** Una QPU puede verse afectada por distintos tipos de ruido. El modelo de ruido de Qiskit Aer utilizado aquí solo simula algunos de ellos y, por lo tanto, es probable que sea menos severo que el ruido de una QPU real.
> 
> Para más detalles sobre qué errores se incluyen al inicializar un modelo de ruido desde una QPU, consulta la referencia de la API de Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Ejemplo con ruido:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Simulación de Clifford
Dado que los Circuits de Clifford pueden simularse eficientemente con resultados verificables, la simulación de Clifford es una herramienta muy útil. Para un ejemplo detallado, consulta [Simulación eficiente de Circuits estabilizadores con primitivas de Qiskit Aer](/guides/simulate-stabilizer-circuits).

Ejemplo:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Próximos pasos
> **Tip:** - Revisa ejemplos detallados de [primitivas.](/guides/primitives-examples)
>     - Lee [Migrar a primitivas V2](/guides/v2-primitives).
>     - Practica con primitivas siguiendo la [lección de función de costo](/learning/courses/variational-algorithm-design/cost-functions) en IBM Quantum Learning.
>     - Aprende a transpilar localmente en la sección [Transpile](/guides/transpile).
>     - Prueba el tutorial [Comparar configuraciones del Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).